# 第 5 章 · 准备"考题"：一万例真实病例

**为什么要做**：要知道 agent 准不准，先得有一套考题：真实发表过的罕见病病例，每例都有症状和确诊的答案。
考题还必须可靠：每个人下载到的完全一样；而且要留一部分到最后才用，防止边考边改。

**做完得到什么**：
- 一条命令下载一万多例公开病例，并自动核对文件没被改过
- 每例整理成统一的格式
- 按论文分成两半：一半平时用，一半最后才考

从上往下一格一格点 ▶。

In [1]:
# 进到项目目录（这格不用看懂）
import os
import subprocess

os.chdir(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
os.environ.update(NO_COLOR="1", TERM="dumb", GIT_PAGER="cat", PAGER="cat",
                  GH_SPINNER_DISABLED="1", GH_PAGER="cat")
print("已进入项目目录")

已进入项目目录


## 1 · 下载并核对

**为什么**：数据不放进代码记录里（太大），但必须保证每个人拿到的一模一样。做法是记下文件的"指纹"，下载后核对。

In [2]:
!uv run raredx data fetch

ok  data/raw/phenopacket-store/0.1.27/all_phenopackets.zip  (release 0.1.27, sha256 verified)


👀 `sha256 verified` = 指纹对上了，和官方发布的文件一个字节都不差。已经下载过的话，只核对、不重新下载。

## 2 · 考题有多少

**为什么**：先知道手上有什么，再谈怎么考。

In [3]:
!uv run raredx data stats

cases                                    10374


skipped                                  {'no_phenotype': 3}
with a causal gene                       10305
distinct diseases (OMIM)                 780
distinct genes                           699
distinct source papers (PMID)            1733
HPO terms per case, median (min–max)     7 (1–66)
split design / heldout                   6014 / 4360


👀 10,374 例，780 种病，699 个基因，来自 1,733 篇论文。

最后一行：平时用 6,014 例，最后才考 4,360 例。**按论文分，不按病例分**：同一篇论文里的病人常是一家人，
一个放平时用、一个放最后考，就等于提前看过答案。所以一篇论文的病人只能在同一边。

## 3 · 一道题长什么样

**为什么**：看一例具体的，就知道 agent 要做的是什么。

In [4]:
from pathlib import Path

from raredx.data import phenopackets as pp

cases, _ = pp.load(pp.archive_path(Path("data")))
c = next(c for c in cases if c.id == "PMID_39701103_Proband_F3")

print("症状（题目）：")
for label in c.hpo_labels:
    print("  -", label)
print("\n确诊（答案）：", c.disease_label, f"（{c.disease_id}）")
print("致病基因：  ", c.gene)
print("来源论文：  ", c.pmid)
print("分在：      ", "平时用" if c.split == "design" else "最后才考")

症状（题目）：
  - Primary amenorrhea
  - Premature ovarian insufficiency
  - Bilateral sensorineural hearing impairment
  - Intellectual disability, mild
  - Global developmental delay

确诊（答案）： Perrault syndrome 7 （OMIM:621101）
致病基因：   DAP3
来源论文：   PMID:39701103
分在：       平时用


👀 题目是症状：听力下降 + 卵巢功能不全 + 发育迟缓。答案是 Perrault 综合征。
agent 要做的，就是只看症状，说出这个病和基因。

还有一点：**答案就写在来源论文里**。以后 agent 去查文献时，必须把这篇论文挡住，否则就是偷看答案。

## 要记住的

| | 是什么 | 公司为什么要它 |
|---|---|---|
| 评测集 | 一套带标准答案的考题 | 没有它，"准不准"只能靠感觉 |
| 指纹（sha256） | 文件的唯一标识 | 保证每个人用的数据完全一样 |
| 开发集 / 留出集 | 平时用的一半 / 最后才考的一半 | 防止边考边改、分数虚高 |
| 防泄漏 | 不让 agent 看到答案的来源 | 否则分数是假的 |

**面试时可以这么说**："评测集是一万例公开病例，下载带校验；按论文切分开发集和留出集，检索时屏蔽来源论文，防止答案泄漏。"